# 04 — Structural-Slope Invariance and Demographic Robustness

**Input:** `composite_scores.csv` from notebook 3, `meta.csv` from notebook 1.

This tests the study's central claim precisely: not just whether usage, level,
and field have *some* relationship with CTE (an additive covariate test), but
whether the AIL-to-CTE and MMR-to-CTE associations *themselves* change with
usage level (a structural-slope test). Both are needed — the additive test alone
does not establish usage-independence of the slopes.

**Note:** field-of-study categories are mapped explicitly to three canonical
buckets (Computing / Other STEM / Non-STEM) rather than with a lossy
`startswith()` fallback, and the mapping asserts that nothing is left unmapped.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

comp = pd.read_csv("composite_scores.csv", index_col=0)
meta = pd.read_csv("meta.csv", index_col=0)
for c in ["AIU", "CO", "AIL", "MMR"]:
    comp[c + "_c"] = comp[c] - comp[c].mean()
print("N =", len(comp))
print("\nRaw field_of_study categories in this data:")
print(meta["field_of_study"].value_counts())

## Test 1 — Additive covariate block (usage, level, field)

In [ ]:
comp2 = comp.join(meta[["level_of_study", "field_of_study", "usage_time"]])
comp2["usage_num"] = comp2["usage_time"].map({
    "I do not use AI": 0, "Less than 30 minutes": 1, "30 minutes – 1 hour": 2,
    "1–2 hours": 3, "2–4 hours": 4, "More than 4 hours": 5})

comp2["level_short"] = comp2["level_of_study"].map(
    {"Undergraduate": "UG", "Postgraduate": "PG", "Integrated / Dual Degree": "INT"})

# Explicit mapping for all three canonical categories -- no startswith() fallback.
# Unmapped categories surface as NaN below rather than being silently absorbed.
FIELD_MAP = {
    "Computing (e.g., Computer Science, Information Technology, Data Science, Artificial Intelligence, Software Engineering)": "Computing",
    "Non-STEM (e.g., Commerce, Economics, Management, Psychology, Arts, Law, Education, Journalism, Design)": "NonSTEM",
    "Other STEM (e.g., Engineering, Mathematics, Statistics, Physics, Chemistry, Biology, Environmental Science)": "OtherSTEM",
}
comp2["field_short"] = comp2["field_of_study"].map(FIELD_MAP)
n_unmapped = comp2["field_short"].isna().sum()
print(f"Unmapped field_of_study values: {n_unmapped}")
if n_unmapped > 0:
    print("Unmapped categories found -- fix FIELD_MAP above before proceeding:")
    print(comp2.loc[comp2["field_short"].isna(), "field_of_study"].unique())
assert n_unmapped == 0, "Unmapped field_of_study categories present -- do not proceed silently."

dummies = pd.get_dummies(comp2[["level_short", "field_short"]], drop_first=True).astype(float)
y = comp2["CTE"].values
X_base = sm.add_constant(comp2[["AIU_c", "CO_c", "AIL_c", "MMR_c"]].values)
X_full = np.column_stack([X_base, comp2[["usage_num"]].values, dummies.values])
base = sm.OLS(y, X_base).fit()
full = sm.OLS(y, X_full).fit()
ftest = full.compare_f_test(base)
print(f"\nCovariate block: F={ftest[0]:.3f}  df={ftest[2]}  p={ftest[1]:.4f}\n")

names = ["const", "AIU_c", "CO_c", "AIL_c", "MMR_c", "usage_num"] + list(dummies.columns)
print("Individual covariate effects (to identify what is/isn't driving the block result):")
for i, name in enumerate(names):
    print(f"  {name:<22} coef={full.params[i]:+.4f}  p={full.pvalues[i]:.4f}")

**Interpretation:** if the block is significant, check which individual term is
driving it. If `usage_num` itself is non-significant while a demographic term is
significant, the usage-independence claim is unaffected; report the demographic
effect as a separate, incidental finding.

## Test 2 — Structural-slope invariance (the direct test of usage-independence)

In [ ]:
comp["AIL_AIU"] = comp["AIL_c"] * comp["AIU_c"]
comp["MMR_AIU"] = comp["MMR_c"] * comp["AIU_c"]

m_base = smf.ols("CTE ~ AIU_c + CO_c + AIL_c + MMR_c", comp).fit()
m_slope = smf.ols("CTE ~ AIU_c + CO_c + AIL_c + MMR_c + AIL_AIU + MMR_AIU", comp).fit()

print(f"AIL x AIU: b={m_slope.params['AIL_AIU']:.4f}  p={m_slope.pvalues['AIL_AIU']:.4f}")
print(f"MMR x AIU: b={m_slope.params['MMR_AIU']:.4f}  p={m_slope.pvalues['MMR_AIU']:.4f}")

ftest2 = m_slope.compare_f_test(m_base)
print(f"\nJoint test: F={ftest2[0]:.3f}  p={ftest2[1]:.4f}  "
      f"(delta R2 = {m_slope.rsquared - m_base.rsquared:.4f})")

**Interpretation:** a non-significant joint test here means the strength of the
AIL-to-CTE and MMR-to-CTE associations does not change with usage level — this is
the specific, direct evidence needed to describe the relationship as
usage-independent, rather than inferring it indirectly from Test 1 alone.

**Note on scope:** this tests whether a single structural-slope model, fit to the
whole sample with usage as a continuous moderator, shows evidence of
usage-dependence. It does not test classical multi-group measurement invariance
across discrete usage groups, which this design deliberately avoids to prevent
an arbitrary and information-losing cutoff.

**Next notebook:** `05_person_centered_profiles.ipynb`